# 11.3 參數傳遞機制與副作用防禦（不可變物件 vs 可變物件串列/字典傳遞）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_11-3_parameter_passing_and_side_effects.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**先備知識**：已掌握 11.1 函數基本定義與呼叫、11.2 函數回傳值 `return`，以及第 8 章串列（List）、第 10 章字典（Dict）基本操作。

---

### 學習導覽：解開函數傳參的「靈異現象」

在學習 Python 函數時，許多初學同學常會遇到一個令人百思不得其解的現象：
- 為什麼我在函數裡面把傳進來的整數數值 `x` 加上 10，回到主程式看，外部的變數卻「絲毫沒有改變」？
- 可是，為什麼我把一個串列（List）傳進函數，在函數內部執行了 `.append()`，回到主程式一看，外部的串列竟然「同步被竄改」了？

這不是靈異現象，而是 Python 核心的**「物件參照傳遞（Pass-by-assignment / Pass-by-object-reference）」**機制！

在本單元中，我們將透過生活化比喻與微型階梯，徹底剖析變數傳入函數後的底層記憶體行為，並建立強大的防禦性編程思維：
1. **11.3.1 不可變物件傳遞**：整數、浮點數、字串與元組——「分身名牌」與數值獨立性。
2. **11.3.2 可變物件傳遞**：串列與字典——「共用鑰匙」引發的外部連動與副作用（Side Effects）。
3. **11.3.3 賦值遮蔽 vs 就地修改**：解密 `arr = []`（換鑰匙斷線）與 `arr.append()`（原處動刀）的本質差異。
4. **11.3.4 防禦性複製**：在函數內建立 `.copy()` 防火牆，既享受函數封裝，又絕不污染外部原始資料。
5. **11.3.5 參數預設值語法**：掌握預設引數的定義規範（先無後有原則）與呼叫端彈性。
6. **11.3.6 經典陷阱防範**：破除 Python 史上最大坑「可變物件作為預設參數」，學會黃金標準 `None` 慣用法。

讓我們一步一腳印，徹底馴服函數參數傳遞，寫出安全、穩健且具備 APCS 實戰防禦力的優質程式碼！

### 11.3.1 不可變物件傳遞（整數、字串、元組）：函數內部修改為何不影響外部？

#### 1. 生活故事比喻：影印本上的塗鴉
想像老師發給全班一張寫著數字 100 的數學作業影印本（不可變物件如整數、浮點數、字串、元組）。你在自己的影印本上拿立可白塗掉，改寫成 150。請問，放回老師講桌上的那張原始母稿，數字會變成 150 嗎？當然不會！你在影印本上的任何塗改，都只屬於你個人的臨時筆記，完全動搖不了原始的母稿。

#### 2. 底層運作機制：物件參照與重新綁定（Rebinding）
在 Python 中，所有變數都是貼在記憶體物件上的「名牌貼紙」。當我們呼叫 `modify(x)` 並將整數變數 `score = 100` 傳入函數時：
- 剛進入函數的瞬間，函數內部的參數名牌 `num` 和外部的名牌 `score` 確實短暫指向同一個記憶體中的數值 `100`。
- 但是，**整數是不可變物件（Immutable）**，電腦記憶體中那個數值 `100` 的方塊一旦建立就永遠無法被就地修改！
- 當函數內部執行 `num = num + 50` 時，Python 會在記憶體中計算出一個全新的數值物件 `150`，並將參數名牌 `num` 撕下來，貼到這個全新的 `150` 上。
- 此時，外部的主程式名牌 `score` 依然牢牢貼在原本的 `100` 上，兩者從此各走各的路，互不相干！

#### 3. 初學者常見陷阱與觀念澄清
很多初學同學會以為寫了函數：
```python
def add_ten(n):
    n = n + 10

val = 5
add_ten(val)
# 誤以為 val 會變成 15！
```
請務必記住：在 Python 中，你**絕對不可能**透過在函數內部對不可變參數進行重新賦值（`n = ...`），來改變外部變數的值！如果你希望外部變數更新，唯一的正途就是**透過 `return` 把計算結果送出來**，再由外部變數重新接收：`val = add_ten(val)`。

#### 4. APCS 實戰提示
在 APCS 競賽中，當你需要設計處理單一數值、旗標狀態或座標數值的輔助函數時，請放心地傳入數字與字串，函數內部即使做了各種運算推導，也絕對不會破壞外部變數的原始數值，這提供了天然的安全邊界。

In [ ]:
# 範例 11.3.1：不可變物件（整數、字串）傳入函數後的獨立性驗證

def try_to_modify(num, text):
    print(f"  [函數內部-修改前] num = {num}, text = '{text}'")
    # 進行運算並重新賦值
    num = num + 50
    text = text + " (已更新)"
    print(f"  [函數內部-修改後] num = {num}, text = '{text}'")

# 主程式定義原始變數
original_num = 100
original_text = "原始報告"

print(f"[主程式-呼叫前] original_num = {original_num}, original_text = '{original_text}'")
print("-" * 50)

# 呼叫函數
try_to_modify(original_num, original_text)

print("-" * 50)
print(f"[主程式-呼叫後] original_num = {original_num}, original_text = '{original_text}'")
print("結論：外部不可變變數完全沒有受到函數內部修改的影響！")

In [ ]:
# ==========================================
# [3] Code 填空題 11.3.1
# 任務說明：
# 小華想寫一個調薪函數 `give_raise(salary, bonus)`。
# 他希望計算加薪後的總薪資，但發現直接在函數內修改參數無法改變外部變數。
# 請補齊程式碼中的挖空處 `___`，讓函數透過 return 回傳新薪資，
# 並在主程式中正確接收更新後的數值。
# ==========================================

def give_raise(salary, bonus):
    new_salary = salary + bonus
    # 提示：必須透過 return 送出結果
    return ___

# 主程式測試
current_salary = 40000
monthly_bonus = 5000

# 提示：外部變數必須接收函數的回傳值
current_salary = give_raise(___, ___)

print(f"調薪後的薪資為: {current_salary}")  # 預期輸出: 45000

In [ ]:
# ==========================================
# [4] Code 練習題 11.3.1
# 任務說明：
# 請設計一個名為 `apply_discount(price, rate)` 的函數：
# 1. 接收商品原價 price（整數）與折扣率 rate（浮點數，如 0.8 代表 8 折）。
# 2. 在函數內計算折後金額並取整數（使用 int() 捨去小數）。
# 3. 回傳計算後的折後價。
# 4. 請在主程式中呼叫此函數，並印出計算結果與原價格，驗證原價格不受影響。
#
# 【公開測試資料 1】
# 呼叫：apply_discount(1000, 0.8)
# 預期輸出：
# 折後價: 800
# 原價保持: 1000
#
# 【公開測試資料 2】
# 呼叫：apply_discount(250, 0.75)
# 預期輸出：
# 折後價: 187
# 原價保持: 250
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.3.1
# 任務說明：
# 撰寫一個安全字串加密函數 `caesar_shift(text, shift)`：
# 1. 傳入一個純大寫英文字串 text 與一個位移量 shift（整數）。
# 2. 函數內部逐一走訪字元，利用 ord() 與 chr() 將每個字母往後位移 shift 個位置
#    （若超過 'Z' 則循環回到 'A'，即 (ord(c) - ord('A') + shift) % 26 + ord('A')）。
# 3. 函數回傳加密後的全新字串。
# 4. 主程式中傳入原始字串，印出加密結果，並嚴格確認原始字串變數完好如初。
#
# 注意：無公開測試資料，請發揮獨立思維，自訂測試字串驗證！
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

### 11.3.2 可變物件傳遞（串列、字典）：內部呼叫方法導致外部同步變化的現象

#### 1. 生活故事比喻：共用同一個大置物櫃的鑰匙
想像你和好朋友在公共置物中心租了一個大置物櫃（可變物件，如串列 List 或字典 Dict）。置物櫃裡放著你們一起收集的球卡。今天你把置物櫃的備份鑰匙交給好朋友（將串列作為參數傳入函數）。好朋友拿著鑰匙打開置物櫃，往裡面放了一張限量版金卡（執行了 `scores.append("Gold")`）。當好朋友離開後，你拿著自己的原始鑰匙去打開置物櫃，你會看到那張金卡嗎？百分之百會看到！因為你們操作的是「同一個實體置物櫃」！

#### 2. 底層運作機制：共享參照（Shared Reference）與副作用（Side Effect）
在 Python 中，串列（List）和字典（Dict）是**可變物件（Mutable）**。
- 當我們將串列 `my_list = [10, 20]` 傳入函數 `def process(arr):` 時，參數 `arr` 和外部的 `my_list` 拿到的是同一個記憶體位址（共用同一把鑰匙）。
- 當函數內部呼叫可變物件的「就地修改方法」，例如：
  - 串列：`arr.append(x)`、`arr.pop()`、`arr.remove(x)`、`arr[0] = 99`
  - 字典：`d[key] = value`、`del d[key]`
- 電腦會直接沿著這個記憶體位址，走到原本那個串列或字典所在的記憶體區域，在原地直接動手術！
- 由於記憶體中的實體物件已經被改動，外部變數 `my_list` 讀取時，自然會「看到改動後的結果」。這種函數內部修改造成外部資料狀態同步改變的現象，在電腦科學中被稱為**「副作用（Side Effect）」**。

#### 3. 初學者常見困惑與除錯警訊
許多初學者在寫程式時，常常隨手將串列傳入一個「只想算算統計資料」的函數，結果該函數內部順便寫了一句 `arr.pop(0)`，回到主程式後，初學者赫然發現自己的資料竟然少了第一筆，導致後續所有演算法全數失控崩潰！這往往是競賽中最難排查的臭蟲之一。

#### 4. APCS 實戰視角：雙面刃的運用
在 APCS 考場上，可變物件的副作用是一把雙面刃：
- **正面優勢（極致省時省記憶體）**：當處理大型地圖矩陣（如 1000x1000）或圖論鄰接串列時，函數直接原地修改傳入的串列，避免了複製百萬資料的龐大時間與記憶體開銷（避免引發 TLE 與 MLE）。
- **負面隱患（破壞原始測資）**：如果該題後續還需要使用原始測資順序，原處修改會徹底毀掉原始資料。因此，我們必須清楚意識到自己何時在「故意原地修改」，何時需要「防禦隔離」。

In [ ]:
# 範例 11.3.2：可變物件（串列與字典）傳入函數後的連動修改現象

def add_extra_bonus(scores_list, player_info):
    print("  [函數內部] 正在對傳入的串列與字典進行就地修改...")
    # 串列就地追加元素
    scores_list.append(100)
    # 字典就地新增鍵值對
    player_info["level"] = "Elite"
    print(f"  [函數內部] scores_list = {scores_list}")
    print(f"  [函數內部] player_info = {player_info}")

# 主程式定義外部資料
my_scores = [85, 90, 95]
user = {"name": "Alice", "level": "Novice"}

print(f"[主程式-呼叫前] my_scores = {my_scores}")
print(f"[主程式-呼叫前] user = {user}")
print("-" * 55)

# 呼叫函數（傳入串列與字典）
add_extra_bonus(my_scores, user)

print("-" * 55)
print(f"[主程式-呼叫後] my_scores = {my_scores}")
print(f"[主程式-呼叫後] user = {user}")
print("震撼結論：外部的串列與字典同步被修改了！這就是可變物件的副作用現象。")

In [ ]:
# ==========================================
# [3] Code 填空題 11.3.2
# 任務說明：
# 某社團主辦活動，想透過函數 `check_in(attendees, student_id)` 進行簽到。
# 請補齊程式碼中的 `___`，使用串列的 `.append()` 方法就地將學生 ID 加入簽到簿中，
# 並在主程式中驗證簽到簿是否已成功在原地更新。
# ==========================================

def check_in(attendees, student_id):
    # 提示：直接使用串列的追加方法進行原地登記
    attendees.___(student_id)
    print(f"成功登記學生: {student_id}")

# 主程式測試
signed_list = [101, 102]

print(f"簽到前名冊: {signed_list}")

# 呼叫簽到函數
check_in(signed_list, ___)

print(f"簽到後名冊: {signed_list}")  # 預期輸出包含 103

In [ ]:
# ==========================================
# [4] Code 練習題 11.3.2
# 任務說明：
# 請設計一個名為 `update_inventory(stock_dict, item, amount)` 的函數：
# 1. 接收庫存字典 stock_dict、商品名稱 item（字串）與進貨數量 amount（整數）。
# 2. 如果商品已在字典中，將庫存量累加 amount；若不在字典中，將其初值設為 amount。
# 3. 此函數無需 return，而是利用可變物件的原處修改特性，直接更新外部傳入的字典。
# 4. 主程式中呼叫此函數兩次，並印出最終的字典庫存狀況。
#
# 【公開測試資料 1】
# 初始字典：{"apple": 10, "banana": 5}
# 呼叫：update_inventory(stock, "apple", 5)
# 預期輸出：
# 最新庫存: {'apple': 15, 'banana': 5}
#
# 【公開測試資料 2】
# 初始字典：{"apple": 15, "banana": 5}
# 呼叫：update_inventory(stock, "orange", 8)
# 預期輸出：
# 最新庫存: {'apple': 15, 'banana': 5, 'orange': 8}
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.3.2
# 任務說明：
# 請設計一個遊戲血量結算函數 `apply_damage(team_hp_list, damage)`：
# 1. 傳入一個代表隊員血量的整數串列 team_hp_list 與受傷點數 damage。
# 2. 函數內部必須「原地修改」該串列中的每一位隊員血量（將每位隊員血量扣除 damage），
#    若扣除後小於 0 則強制將該隊員血量重置為 0（不可為負數）。
# 3. 函數不使用 return，完全依靠原處修改（In-place modification）。
# 4. 主程式宣告血量串列，呼叫函數後印出串列，驗證外部血量是否確實被扣減且不低於 0。
#
# 注意：無公開測試資料，請自行構思血量數值並執行驗證！
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

### 11.3.3 賦值遮蔽 vs 就地修改：`arr = []`（重新指向）與 `arr.append()`（原處竄改）

#### 1. 生活故事比喻：換掉手上的鑰匙 vs 走進倉庫大掃除
很多初學者在學完 11.3.2 之後，興沖沖地想寫一個「清空外部串列」的函數：
```python
def clear_list(arr):
    arr = []  # 誤以為這樣可以把外部的串列清空！
```
執行後卻驚訝地發現：外部的串列竟然「完全沒有被清空」，依然塞滿了原本的資料！為什麼會這樣？
- **`arr.append()` 或 `arr.clear()`（就地修改）**：就像你拿著置物櫃鑰匙「走進置物櫃裡面」，把裡面的東西丟掉。此時置物櫃本體確實被清空了。
- **`arr = []`（重新賦值）**：就像你「把手上的置物櫃鑰匙丟掉，換拿一張全新的空白名牌」！此時你手上的名牌確實變成了空的，但原本那個置物櫃呢？它依然好端端地立在原地，裡面原本的東西連一粒灰塵都沒少！

#### 2. 底層運作機制：等號 `=` 永遠是「綁定新物件」，絕非修改原物件
在 Python 中，賦值符號 `=` 的本質是**「將左側的變數名稱，重新綁定到右側求出的新物件上」**：
- 當在函數內部執行 `arr = []` 時，Python 在記憶體中建立了一個全新的空串列 `[]`，並將區域變數名牌 `arr` 指向這個新空串列。
- 這一步徹底「切斷」了內部變數 `arr` 與外部原始串列的記憶體連結！
- 既然連結已經斷開，後續你對內部 `arr` 做任何操作，都只發生在這個新建立的空串列上，外部原始串列毫髮無傷。

#### 3. 正確清空與就地替換的技巧
如果你真的希望在函數內部將傳入的外部串列清空，正確的做法有兩種：
1. **呼叫專屬清空方法**：`arr.clear()`（Python 3 推薦，直接清空原記憶體區塊）。
2. **切片賦值神技**：`arr[:] = []`（將原串列所有索引區間替換為空，保留原記憶體位址）。

#### 4. APCS 競賽防坑指南
在 APCS 競賽中，當你需要重置某個走訪記錄矩陣或緩存串列時，請特別分清楚：
- 若寫 `visited = [[False]*C for _ in range(R)]`，這是在建立全新矩陣。
- 若傳入子函數處理，千萬別妄想用 `arr = [0]*N` 來重置外部陣列；如果必須在原地重置，請走訪每個索引給予初值，或使用切片賦值！

In [ ]:
# 範例 11.3.3：重新賦值（斷開連結）vs 就地修改（原處清空）對比

def reset_by_reassign(arr):
    print("  [reassign 內部] 執行 arr = [] (重新指向新物件)")
    arr = []
    print(f"  [reassign 內部] 函數結束時 arr = {arr}")

def reset_by_clear(arr):
    print("  [clear 內部] 執行 arr.clear() (就地清空原本物件)")
    arr.clear()
    print(f"  [clear 內部] 函數結束時 arr = {arr}")

# 實驗組 A：重新賦值
data_A = [10, 20, 30]
print(f"[實驗 A 呼叫前] data_A = {data_A}")
reset_by_reassign(data_A)
print(f"[實驗 A 呼叫後] data_A = {data_A}  --> 驚喜！外部串列完全沒被清空！")
print("-" * 60)

# 實驗組 B：就地清空
data_B = [10, 20, 30]
print(f"[實驗 B 呼叫前] data_B = {data_B}")
reset_by_clear(data_B)
print(f"[實驗 B 呼叫後] data_B = {data_B}  --> 成功！外部串列被真正清空了！")

In [ ]:
# ==========================================
# [3] Code 填空題 11.3.3
# 任務說明：
# 某位同學想寫一個重置分數串列的函數 `reset_scores(scores)`。
# 他原本寫 `scores = [0, 0, 0]` 發現外部完全無效。
# 請補齊程式碼中的 `___`，使用切片賦值 `scores[:] = ...` 或是 `.clear()`，
# 讓外部傳入的串列能夠真正被原地清空。
# ==========================================

def reset_scores(scores):
    # 提示：使用 .clear() 方法就地清空原物件
    scores.___()

# 主程式測試
my_scores = [98, 85, 76]
print(f"重置前: {my_scores}")

reset_scores(my_scores)

print(f"重置後: {my_scores}")  # 預期輸出: []

In [ ]:
# ==========================================
# [4] Code 練習題 11.3.3
# 任務說明：
# 請觀察以下兩種將串列元素翻倍的函數實作：
# 1. 實作函數 `double_by_reassign(nums)`：在內部執行 `nums = [x * 2 for x in nums]`，不回傳。
# 2. 實作函數 `double_in_place(nums)`：在內部使用迴圈與索引 `nums[i] = nums[i] * 2` 原地翻倍。
# 3. 在主程式中分別建立兩組相同內容的串列，測試兩個函數，並印出呼叫後的結果。
#
# 【公開測試資料 1】
# 測試串列：[1, 2, 3]
# 呼叫 double_by_reassign 後外部串列預期輸出：
# [1, 2, 3] (維持不變)
#
# 【公開測試資料 2】
# 測試串列：[1, 2, 3]
# 呼叫 double_in_place 後外部串列預期輸出：
# [2, 4, 6] (成功原地翻倍)
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.3.3
# 任務說明：
# 設計一個安全過濾正數的函數 `filter_positive_in_place(nums)`：
# 1. 傳入一個包含正負數的整數串列 nums。
# 2. 必須利用切片賦值 `nums[:] = [x for x in nums if x > 0]` 原處更新原串列。
# 3. 驗證呼叫該函數後，外部原本的串列是否只剩下正數，且記憶體位址不變
#    （提示：可使用 id(nums) 在呼叫前後印出記憶體編號，驗證 ID 是否完全一致）。
#
# 注意：無公開測試資料，請自行設計正負整數測試串列並驗證！
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

### 11.3.4 防禦性複製：在函數內使用 `.copy()` 阻斷對外部原始陣列的副作用

#### 1. 生活故事比喻：辦公室裡的影印機防護牆
假設老闆交給你一份極具紀念價值的「創社元老簽名名單」（原始串列），交代你：「去把這份名單按照筆畫順序排一下，並把字跡模糊的標記出來，但這張原稿絕對不能有任何破損或筆跡！」你該怎麼做？你絕對不會笨到直接在原稿上拿剪刀剪貼，而是第一時間走到影印機前，影印一張全新的黑白複本（`.copy()`），在複本上剪剪貼貼、做筆記。做完後把複本成果交給老闆，而原稿完好如初地收回保險箱。這在程式設計中稱為**「防禦性複製（Defensive Copying）」**。

#### 2. 底層運作機制：`.copy()` 打造獨立記憶體副本
如果一個函數需要對傳入的可變物件進行排序、剔除或修改，但設計目標是**「不要干擾外部呼叫者的原始資料」**，我們就必須在函數內部築起防火牆：
- 使用 `safe_arr = arr.copy()` 或 `safe_arr = arr[:]`。
- 這會在記憶體中開闢一塊全新的空間，將原本串列中的每個元素複製一份放進去。
- 此時，`safe_arr` 擁有了自己專屬的置物櫃與鑰匙，它與外部傳入的 `arr` 徹底脫鉤！
- 隨後，函數在 `safe_arr` 上愛怎麼排序（`.sort()`）、愛怎麼刪除（`.pop()`）都完全自由，最後再將加工完的 `safe_arr` 透過 `return` 送交給外部。

#### 3. 純函數（Pure Function）的美德
在高品質軟體工程中，一個「理想的函數」應具備以下特質：
1. 給定相同的輸入，永遠回傳相同的輸出。
2. **無任何副作用（No Side Effects）**：不暗中篡改外部變數、不弄髒傳進來的串列。
透過防禦性複製，初學者可以徹底避免「不知哪一行把資料改壞」的幽靈臭蟲，大幅提升程式碼的可預測性與除錯效率。

#### 4. APCS 實戰場景
在 APCS 考題中（例如第 12 章即將學習的各類排序與統計題目），題目經常要求：
- 第一行輸出：原始輸入順序的某些計算。
- 第二行輸出：排序後的最大三項或最小三項。
如果你的排序函數直接在傳入的串列上執行 `nums.sort()`，原本的輸入順序就被永久破壞了，導致第一行的計算邏輯直接報銷！此時，使用 `.copy()` 先複製再排序，是保證滿分的關鍵心法。

In [ ]:
# 範例 11.3.4：防禦性複製（Defensive Copy）保護外部資料範例

def get_top_three_safely(scores):
    # 步驟 1：防禦性複製！複製一份獨立串列，切斷與外部的牽連
    working_copy = scores.copy()
    
    # 步驟 2：在複本上自由地進行原地排序（由大到小）
    working_copy.sort(reverse=True)
    
    # 步驟 3：回傳前三名
    return working_copy[:3]

# 主程式原始資料
exam_scores = [72, 95, 88, 60, 100, 84]

print(f"[呼叫前] 原始成績單: {exam_scores}")
print("-" * 55)

top_three = get_top_three_safely(exam_scores)

print("-" * 55)
print(f"前三名成績: {top_three}")
print(f"[呼叫後] 原始成績單: {exam_scores}")
print("確認：原始成績單順序完全保持原樣，沒有被排序打亂！")

In [ ]:
# ==========================================
# [3] Code 填空題 11.3.4
# 任務說明：
# 某體育比賽計算得分，規則是「去除最高分與最低分後計算總和」。
# 請補齊函數 `trimmed_sum(scores)` 中的挖空處 `___`：
# 先使用 `.copy()` 複製串列，接著在複本中移除最高與最低分，最後回傳總和，
# 確保外部的原始成績串列不受任何影響。
# ==========================================

def trimmed_sum(scores):
    # 提示：第一步製作防禦性複本
    temp_scores = scores.___()
    
    # 在複本上移除最高分與最低分
    temp_scores.remove(max(temp_scores))
    temp_scores.remove(min(temp_scores))
    
    return sum(temp_scores)

# 主程式測試
raw_scores = [10, 85, 90, 78, 100]
print(f"原始選手評分: {raw_scores}")

result = trimmed_sum(raw_scores)

print(f"去除極端值後總和: {result}")      # 預期輸出: 253 (85+90+78)
print(f"檢查原始資料長度: {len(raw_scores)}")  # 預期長度依然為 5，證明無副作用

In [ ]:
# ==========================================
# [4] Code 練習題 11.3.4
# 任務說明：
# 請設計一個名為 `remove_negatives_safe(nums)` 的函數：
# 1. 接收一個包含正負數的整數串列 nums。
# 2. 在內部製作 nums 的複本，將複本中所有小於 0 的負數移除（或以生成式過濾非負數）。
# 3. 回傳僅含非負數的全新串列。
# 4. 在主程式中驗證：回傳的串列已過濾負數，而傳入的原始串列內容與長度完全不變。
#
# 【公開測試資料 1】
# 傳入：[5, -2, 10, -8, 7]
# 預期輸出：
# 過濾結果: [5, 10, 7]
# 原始串列保持: [5, -2, 10, -8, 7]
#
# 【公開測試資料 2】
# 傳入：[-1, -2, -3]
# 預期輸出：
# 過濾結果: []
# 原始串列保持: [-1, -2, -3]
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.3.4
# 任務說明：
# 請設計一個字典防禦性處理函數 `apply_store_discount(catalog, discount_rate)`：
# 1. catalog 是一個包含商品與價格的字典，如 `{"書包": 800, "鋼筆": 200}`。
# 2. 函數內部必須使用 `catalog.copy()` 複製字典，並將複本中的每個商品價格打折取整數。
# 3. 回傳打折後的全新字典。
# 4. 主程式中印出特價字典與原始字典，嚴格驗證原始字典的售價未遭到竄改。
#
# 注意：無公開測試資料，請自行設計商品目錄字典並驗證！
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

### 11.3.5 參數預設值（Default Parameters）基本語法與呼叫端彈性

#### 1. 生活故事比喻：手搖飲點餐的「預設糖冰」
去飲料店買珍珠奶茶時，如果你只對店員說：「我要一杯珍珠奶茶。」店員會很自然地幫你做成「全糖、正常冰」（這是店家的預設值）。但如果你特別指定：「我要一杯珍珠奶茶，微糖、去冰。」店員就會使用你的自訂要求，覆蓋掉預設值。在 Python 函數設計中，**參數預設值（Default Parameter Values）**就是為參數準備的「預設餐點設定」，讓呼叫函數變得無比親切與彈性。

#### 2. 語法結構與「先無預設、後有預設」的排隊鐵律
在定義函數時，只要在參數名稱後加上等號與預設值即可：
```python
def make_drink(item, sugar="全糖", ice="正常冰"):
    return f"{item} ({sugar}, {ice})"
```
⚠️ **語法致命天條**：
所有**沒有預設值的位置參數**必須排在**前面**，所有**有預設值的參數**必須排在**後面**！
- ✅ 合法：`def calc(a, b=10, c=20):`
- ❌ 違法：`def calc(a=10, b):`
如果敢把有預設值的參數排在前面，Python 在定義期就會立刻亮紅燈報警：`SyntaxError: non-default argument follows default argument`！這是因為呼叫時如果只傳入一個引數，電腦根本無法分辨這個引數到底要給誰。

#### 3. 呼叫端的超高自由度
有了參數預設值，呼叫端可以依據需求自由選擇：
1. **全部依賴預設值**：`make_drink("珍奶")` $ightarrow$ 珍奶 (全糖, 正常冰)
2. **覆蓋部分預設值**：`make_drink("珍奶", "半糖")` $ightarrow$ 珍奶 (半糖, 正常冰)
3. **全部自訂指定**：`make_drink("珍奶", "無糖", "去冰")` $ightarrow$ 珍奶 (無糖, 去冰)

#### 4. APCS 競賽實務價值
在 APCS 解題時，我們常會撰寫一些通用的矩陣印出或除錯輔助函數：
例如 `def print_grid(grid, sep=" "):`，平時直接 `print_grid(g)` 使用空格隔開；需要緊密印出時只要 `print_grid(g, "")`。利用預設參數能讓你的自訂競賽工具庫既強大又極度精簡！

In [ ]:
# 範例 11.3.5：參數預設值的定義規範與靈活呼叫

def calculate_fare(distance, base_fee=85, per_km=20):
    """計算計程車車資：起跳價 base_fee，每公里加收 per_km"""
    total = base_fee + distance * per_km
    return total

# 情境 1：完全使用預設的起跳價與每公里單價
fare1 = calculate_fare(5)
print(f"市區基本計費 (5公里): {fare1} 元")  # 85 + 5*20 = 185

# 情境 2：春節加成，自訂起跳價為 105 元（覆蓋 base_fee），每公里維持預設 20 元
fare2 = calculate_fare(5, base_fee=105)
print(f"春節加成計費 (5公里): {fare2} 元")  # 105 + 5*20 = 205

# 情境 3：山區長途特惠，起跳價 100，每公里 15 元（覆蓋所有預設值）
fare3 = calculate_fare(10, 100, 15)
print(f"山區特惠計費 (10公里): {fare3} 元")  # 100 + 10*15 = 250

In [ ]:
# ==========================================
# [3] Code 填空題 11.3.5
# 任務說明：
# 請設計一個文字框產生器 `banner(text, border="*", padding=1)`。
# 預設的外框符號是 `*`，預設的內縮空格數 padding 為 1。
# 請補齊程式碼中的 `___`，完成函數宣告與字串組裝。
# ==========================================

# 提示：注意參數順序，有預設值的參數必須排在後面
def banner(text, border="*", padding=___):
    content = " " * padding + text + " " * padding
    line = border * (len(content) + 2)
    middle = border + content + border
    return f"{line}\n{middle}\n{line}"

# 主程式測試
# 1. 全部採用預設邊框與邊距
print(banner("APCS 滿分"))

# 2. 自訂邊框為 '#'
print(banner("Hello", border="#"))

In [ ]:
# ==========================================
# [4] Code 練習題 11.3.5
# 任務說明：
# 請設計一個名為 `power(base, exp=2)` 的次方計算函數：
# 1. 接收底數 base（數值）與指數 exp（預設值為 2，即預設計算平方）。
# 2. 函數計算並回傳 base 的 exp 次方（使用 ** 運算子）。
# 3. 主程式中測試兩種呼叫方式：
#    - 呼叫方式 A：只傳入一個參數，驗證是否正確計算平方。
#    - 呼叫方式 B：傳入兩個參數，計算自訂次方（如 3 次方）。
#
# 【公開測試資料 1】
# 呼叫：power(7)
# 預期輸出：
# 7 的平方為: 49
#
# 【公開測試資料 2】
# 呼叫：power(2, 5)
# 預期輸出：
# 2 的 5 次方為: 32
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.3.5
# 任務說明：
# 請設計一個多功能會員結帳計算函數 `checkout(amount, discount=0.9, is_vip=False)`：
# 1. 基本金額 amount（整數）。
# 2. 預設享有 9 折折扣（discount=0.9）。
# 3. 若 is_vip 為 True，則在折扣後再折抵 100 元現金（若扣減後金額小於 0 則以 0 計）。
# 4. 最終金額需以整數 int() 回傳。
# 5. 請分別測試「一般民眾預設結帳」、「自訂 8 折結帳」與「VIP 特殊折抵結帳」三種情境。
#
# 注意：無公開測試資料，請自行規劃測試數據與邏輯！
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

### 11.3.6 陷阱防範：避免使用可變物件（如 `[]`）作為預設參數的機制與正解（`None` 慣用法）

#### 1. 生活故事比喻：所有人共用的「幽靈置物箱」
想像一家超市提供「免費置物籃」。規則本來應該是：每當新顧客走進店裡且沒有自備購物袋時，店員就從倉庫拿出一個「全新的空籃子」給他。然而，店員偷懶了，他在櫃檯上只放了一個置物籃（可變物件 `[]`）。第一個客人進來買了蘋果，放進籃子結帳走人，但籃子沒清空；第二個客人進來沒帶袋子，店員把同一個籃子遞給他，第二個客人驚愕地發現：「咦？籃子裡怎麼有上一位客人留下來的爛蘋果？！」這就是 Python 程式設計中最著名的「可變預設參數陷阱」！

#### 2. 為什麼會這樣？底層原理解密：函數定義期（Def Time）的單次求值
初學者直覺以為：每次呼叫 `def add_item(item, basket=[])` 時，Python 就會建立一個全新的空串列。
❌ **大錯特錯！**
- **Python 的預設參數表達式，只在「函數定義期（Def Time）」被計算並建立一次！**
- 當直譯器讀到 `def ... basket=[]` 這行代碼時，記憶體中就只產生了一個唯一的實體空串列。
- 當後續你呼叫 `add_item("蘋果")` 時，因為沒傳引數，`basket` 指向這個唯一的全域共用串列，並在裡面加入了蘋果。
- 當你第二次呼叫 `add_item("香蕉")` 時，`basket` 依然指向那個同一個串列，於是香蕉就被加在蘋果後面，變成了 `["蘋果", "香蕉"]`！
- 每次呼叫的資料全部混在一起，造成嚴重的狀態污染！

#### 3. 黃金解方：不可變的守護神——`None` 慣用法
要如何徹底杜絕這個跨呼叫的資料污染？在 Python 世界中，所有專業工程師都遵守唯一的**黃金標準慣例（Idiom）**：
👉 **永遠使用不可變的 `None` 作為預設值，並在函數內部動態建立新串列！**
```python
def add_item(item, basket=None):
    if basket is None:
        basket = []  # 每次呼叫都是在此時動態產出全新、獨立的空串列！
    basket.append(item)
    return basket
```
- 因為 `None` 是不可變物件，每次呼叫時如果沒有給值，`basket` 就會是 `None`。
- 進入函數內部後，`if basket is None:` 條件觸發，才會在當次呼叫的生命週期中「現場現做」一個全新的空串列 `[]`！
- 如此一來，每一次呼叫都擁有徹底獨立的乾淨容器，再也不會互相污染！

#### 4. APCS 競賽避坑警言
在 APCS 歷屆試題與樹狀走訪、路徑搜尋題型中，我們經常撰寫搜尋函數記錄路徑 `path`。如果你隨手寫成 `def dfs(node, path=[]):`，所有搜尋分支將會共用同一條路徑，讓你的程式碼出現極其詭異且無法重現的死結！請把 `path=None` 刻進你的反射神經中！

In [ ]:
# 範例 11.3.6：可變物件預設參數的致命災難 vs None 標準正解

# ❌ 錯誤示範：使用可變物件 [] 作為預設值（資料交叉污染）
def bad_add_student(name, roster=[]):
    roster.append(name)
    return roster

print("--- 錯誤寫法測試（幽靈共用） ---")
print("第 1 次呼叫（甲班）:", bad_add_student("小明"))  # ['小明']
print("第 2 次呼叫（乙班）:", bad_add_student("小華"))  # 竟然是 ['小明', '小華']！甲班的人跑來乙班了！
print("第 3 次呼叫（丙班）:", bad_add_student("小英"))  # ['小明', '小華', '小英']！徹底崩潰！

print("\n" + "=" * 50 + "\n")

# ✅ 正確示範：使用 None 作為預設值（每次呼叫獨立生成）
def good_add_student(name, roster=None):
    if roster is None:
        roster = []  # 現場為當次呼叫打造專屬的全新空串列
    roster.append(name)
    return roster

print("--- 正確寫法測試（獨立乾淨） ---")
print("第 1 次呼叫（甲班）:", good_add_student("小明"))  # ['小明']
print("第 2 次呼叫（乙班）:", good_add_student("小華"))  # ['小華'] (乾淨獨立！)
print("第 3 次呼叫（丙班）:", good_add_student("小英"))  # ['小英'] (互不干擾！)

In [ ]:
# ==========================================
# [3] Code 填空題 11.3.6
# 任務說明：
# 小林設計了一個購物清單收集器 `create_todo(task, todo_list=None)`。
# 請補齊程式碼中的 `___`，使用 `None` 檢查與動態初始化，
# 徹底防止多次呼叫時待辦事項互相堆疊污染。
# ==========================================

# 提示：預設值設定為不可變的 None
def create_todo(task, todo_list=___):
    # 提示：檢查是否未傳入引數（是否為 None）
    if todo_list is ___:
        todo_list = []  # 現場建立全新的空串列
    
    todo_list.append(task)
    return todo_list

# 主程式測試
user_A_todo = create_todo("寫數學作業")
user_B_todo = create_todo("買牛奶")

print(f"使用者 A 清單: {user_A_todo}")  # 預期輸出: ['寫數學作業']
print(f"使用者 B 清單: {user_B_todo}")  # 預期輸出: ['買牛奶']（不可包含使用者的作業）

In [ ]:
# ==========================================
# [4] Code 練習題 11.3.6
# 任務說明：
# 請設計一個名為 `register_score(name, score, score_dict=None)` 的成績登記函數：
# 1. 參數包含學生姓名 name（字串）、成績 score（整數），以及成績字典 score_dict。
# 2. 預設參數 score_dict 必須為 None。
# 3. 函數內部若發現 score_dict 為 None，必須動態建立一個全新的空字典 `{}`。
# 4. 將學生姓名與成績存入該字典中（`score_dict[name] = score`），並回傳該字典。
# 5. 連續呼叫兩次（均不傳入 score_dict），驗證回傳的兩份字典是否各自獨立，毫無殘留。
#
# 【公開測試資料 1】
# 呼叫：dict1 = register_score("Alice", 95)
# 預期輸出：
# Alice 的獨立字典: {'Alice': 95}
#
# 【公開測試資料 2】
# 呼叫：dict2 = register_score("Bob", 88)
# 預期輸出：
# Bob 的獨立字典: {'Bob': 88}
# （確認 dict2 絕對不能含有 Alice 的成績！）
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.3.6
# 任務說明：
# 請設計一個具有預設清單功能的文字收集器 `collect_words(word, container=None)`：
# 1. 若呼叫時沒有傳入 container，自動建立獨立串列，將 word 加入並回傳。
# 2. 若呼叫時「明確傳入」了某個外部串列（例如 `my_box = ["已存在項目"]`），
#    則直接將 word 加入該外部串列並回傳（此時允許正常的原處追加）。
# 3. 請撰寫測試程式，分別展示：
#    - 兩次不帶 container 呼叫，驗證結果各自獨立。
#    - 一次帶入自訂串列呼叫，驗證自訂串列成功被追加內容。
#
# 注意：無公開測試資料，請自行發揮邏輯完整測試！
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

## 11.3 單元重點回顧與自我檢驗

恭喜你完成第 11.3 單元的學習！參數傳遞與副作用防禦是區分「程式新手」與「成熟工程師」的重要分水嶺。掌握了底層記憶體行為，未來你在撰寫龐大演算法時，就不會再被幽靈般的資料竄改所困擾。

### 核心觀念複習清單
1. **不可變物件傳遞（Pass-by-assignment）**：
   - 包含整數（int）、浮點數（float）、字串（str）與元組（tuple）。
   - 函數內部對參數重新賦值只會改變區域變數的指向，**絕對不會影響外部變數**。
   - 若要更新外部數值，必須依賴 `return` 將新值回傳接收。
2. **可變物件傳遞（副作用 Side Effects）**：
   - 包含串列（list）與字典（dict）。
   - 傳入的是記憶體參照（共用鑰匙），在內部呼叫 `.append()`、`[key]=val` 等原地操作，**外部會同步感知被修改**。
3. **重新賦值 vs 就地修改**：
   - `arr = []` 是切斷連結、指向新物件，外部串列絲毫未動。
   - `arr.clear()` 或 `arr[:] = []` 才是真正原地清空外部資料。
4. **防禦性複製（Defensive Copying）**：
   - 當需要排序或過濾且不希望破壞外部資料時，函數內部第一步先使用 `.copy()` 築起防火牆。
5. **參數預設值規範**：
   - 必須遵守「先無預設、後有預設」的排隊鐵律，否則引發 `SyntaxError`。
6. **可變預設參數陷阱與 `None` 慣用法**：
   - 嚴禁寫 `def f(arr=[]):`，因為預設參數在定義期只建立一次，會造成跨呼叫資料污染。
   - 黃金解法：預設參數設為 `None`，函數內部使用 `if arr is None: arr = []` 動態初始化。

---

### 下一步精彩預告
搞清楚了參數在函數「內部與外部」的資料交接機制後，下一個核心問題來了：
- 變數在函數裡面建立，能在外面使用嗎？
- 如果函數內部宣告了一個名字也叫 `x` 的變數，它會跟外面的全域變數 `x` 打架嗎？
- 為什麼常常看到競賽選手被 `UnboundLocalError` 氣到崩潰？

請緊接著邁向 **[11.4 變數作用域：區域變數（Local Scope）與同名變數遮蔽（Shadowing）]**，解鎖變數生命週期的奧秘！